# ALFRED 半导体生产者价格历史版本来源核查

仅按冻结 `plan.json` 下载 BLS 半导体制造业 PPI 的 93 个 ALFRED 日期版本；不读 SOXX 收益，不训练模型。首次版本与后续修订必须分开。


In [ ]:
from pathlib import Path
import hashlib, io, json, zipfile
import requests
from bs4 import BeautifulSoup
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root: raise RuntimeError("workspace root missing")
    root=root.parent
card=root/"research/semiconductor_ppi_alfred_source_20261022"
plan=json.loads((card/"plan.json").read_text())
assert plan["frozen"] and plan["execution_authorized"]
assert not (card/"retrieval_receipt.json").exists()
sha=lambda b:hashlib.sha256(b).hexdigest()
session=requests.Session()
response=session.get(plan["source_url"],timeout=30)
response.raise_for_status()
soup=BeautifulSoup(response.content,"html.parser")
form=next(f for f in soup.find_all("form") if f.find(attrs={"name":"form[file_type]"}) is not None)
choices=[x.get("value") for x in form.find(attrs={"name":"form[selected_vintage_dates][]"}).find_all("option")]
dates=[d for d in choices if plan["vintage_start"]<=d<=plan["vintage_end"]]
assert len(dates)==plan["expected_selected_vintages"]
assert dates[0]==plan["preflight_form_observation"]["first_selected"]
assert dates[-1]==plan["preflight_form_observation"]["last_selected"]
assert form.find(attrs={"name":"form[obs_end_date]"}).get("value")>=plan["observation_end"]
data=[("form[units]","lin"),("form[obs_start_date]",plan["observation_start"]),
    ("form[obs_end_date]",plan["observation_end"]),("form[entered_vintage_dates]",""),
    ("form[file_type]",plan["download_file_type"]),("form[file_format]",plan["download_file_format"]),
    ("form[download_data]","")]
data += [("form[selected_vintage_dates][]",d) for d in dates]
data += [(h.get("name"),h.get("value","")) for h in form.find_all("input",attrs={"type":"hidden"})]
result=session.post(plan["source_url"],data=data,timeout=60)
result.raise_for_status()
body=result.content
assert len(body)<=plan["request_budget"]["max_archive_bytes"]
assert zipfile.is_zipfile(io.BytesIO(body))
raw=root/plan["raw_path"]
raw.parent.mkdir(parents=True,exist_ok=True)
assert not raw.exists()
raw.write_bytes(body)
archive=zipfile.ZipFile(io.BytesIO(body))
names=archive.namelist()
receipt={"schema_version":1,"status":"source_archive_downloaded_schema_qualification_pending",
    "plan_sha256":sha((card/"plan.json").read_bytes()),"source_url":plan["source_url"],
    "selected_vintage_dates":len(dates),"first_vintage":dates[0],"last_vintage":dates[-1],
    "form_sha256":sha(response.content),"raw_path":plan["raw_path"],"raw_sha256":sha(body),
    "raw_bytes":len(body),"archive_entries":names,"market_outcome_reads":0,"model_fits":0,"accounts":0}
(card/"retrieval_receipt.json").write_text(json.dumps(receipt,indent=2)+"\n")
print(json.dumps({"status":receipt["status"],"vintages":len(dates),"archive_entries":names,"raw_bytes":len(body)}))


In [ ]:
from pathlib import Path
import os, io, json, hashlib, zipfile
import numpy as np
import pandas as pd
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE","true")
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root: raise RuntimeError("workspace root missing")
    root=root.parent
card=root/"research/semiconductor_ppi_alfred_source_20261022"
plan=json.loads((card/"source_quality_plan.json").read_text())
intake=json.loads((card/"plan.json").read_text())
receipt=json.loads((card/"retrieval_receipt.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["execution_authorized"] and not (card/"summary.json").exists()
assert sha(card/"plan.json")==plan["source_plan_sha256"]
assert sha(card/"retrieval_receipt.json")==plan["retrieval_receipt_sha256"]
assert receipt["raw_sha256"]==plan["raw_zip_sha256"]
for relative,digest in plan["provider_files_sha256"].items():
    assert sha(root/relative)==digest,relative
raw=root/receipt["raw_path"]
assert sha(raw)==plan["raw_zip_sha256"]
with zipfile.ZipFile(raw) as archive:
    assert set(archive.namelist())=={"README.txt",plan["csv_name"]}
    readme=archive.read("README.txt");csv_bytes=archive.read(plan["csv_name"])
assert hashlib.sha256(readme).hexdigest()==plan["readme_sha256"]
assert hashlib.sha256(csv_bytes).hexdigest()==plan["csv_sha256"]
text=readme.decode()
assert "Series ID: "+plan["series_id"] in text
assert "Output Format: Observations by Vintage Date, All Observations" in text
frame=pd.read_csv(io.BytesIO(csv_bytes))
assert list(frame.shape)==plan["expected_shape"]
assert frame.columns[0]=="observation_date"
assert len(frame.columns)-1==plan["expected_vintage_dates"]
observation=pd.DatetimeIndex(pd.to_datetime(frame.observation_date))
assert observation.equals(pd.date_range(observation[0],observation[-1],freq="MS"))
assert str(observation[0].date())==intake["observation_start"]
assert str(observation[-1].date())==intake["observation_end"]
vintage_dates=[]
for name in frame.columns[1:]:
    prefix=plan["series_id"]+"_"
    assert name.startswith(prefix) and len(name)==len(prefix)+8
    vintage_dates.append(pd.Timestamp(name[len(prefix):]))
assert len(set(vintage_dates))==len(vintage_dates) and vintage_dates==sorted(vintage_dates)
assert vintage_dates[0].date().isoformat()==intake["preflight_form_observation"]["first_selected"]
assert vintage_dates[-1].date().isoformat()==intake["preflight_form_observation"]["last_selected"]
values=frame.iloc[:,1:].apply(pd.to_numeric,errors="coerce")
assert (values.stack()>0).all()
release_rows=[]
for name,date in zip(values.columns,vintage_dates):
    series=values[name]
    valid=np.flatnonzero(series.notna().to_numpy())
    assert len(valid)>0
    loc=int(valid[-1]);latest_month=observation[loc]
    latest=float(series.iloc[loc])
    previous=float(series.iloc[loc-1]) if loc>=1 and pd.notna(series.iloc[loc-1]) else None
    year_ago=float(series.iloc[loc-12]) if loc>=12 and pd.notna(series.iloc[loc-12]) else None
    release_rows.append({"vintage_date":date,"latest_observation_month":latest_month,
        "latest_value":latest,"previous_month_value_same_vintage":previous,
        "year_ago_value_same_vintage":year_ago,
        "mom_pct_same_vintage":100*(latest/previous-1) if previous else None,
        "yoy_pct_same_vintage":100*(latest/year_ago-1) if year_ago else None})
release=pd.DataFrame(release_rows)
latest_months=pd.PeriodIndex(release.latest_observation_month,freq="M")
expected=pd.period_range(str(latest_months.min()),str(latest_months.max()),freq="M")
missing_latest=[str(x) for x in expected if x not in latest_months]
assert missing_latest==[plan["expected_latest_month_gap"]]
assert release.latest_observation_month.is_unique
assert release.latest_value.notna().all() and release.previous_month_value_same_vintage.notna().all()
assert release.yoy_pct_same_vintage.notna().sum()==len(release)-1
# Qlib supplies the session calendar. No price field or SOXX outcome is read.
import qlib
from qlib.data import D
from qlib.workflow import R
qlib.init(provider_uri=str(root/plan["price_calendar_provider"]),region="us",kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time="2019-01-01",end_time="2026-10-21",freq="day")).normalize()
assert calendar.is_unique and calendar.is_monotonic_increasing
available=[]
for date in release.vintage_date:
    pos=calendar.searchsorted(date,side="right")
    assert pos<len(calendar)
    available.append(calendar[pos])
release["next_qlib_session_0946_NY"]=available
assert (release.next_qlib_session_0946_NY>release.vintage_date).all()
shutdown=release.loc[release.vintage_date.eq(pd.Timestamp("2026-01-14"))].iloc[0]
assert shutdown.latest_observation_month==pd.Timestamp("2025-11-01")
assert pd.notna(values.loc[(observation==pd.Timestamp("2025-10-01")),plan["series_id"]+"_20260114"].iloc[0])
assert values.loc[(observation==pd.Timestamp("2025-10-01")),plan["series_id"]+"_20251125"].isna().iloc[0]
# All revisions below are source-only descriptive; no value is used as a market label.
revised_periods=0
for row in values.itertuples(index=False,name=None):
    observed_values=np.asarray([x for x in row if pd.notna(x)],dtype=float)
    if len(observed_values)>1 and not np.allclose(observed_values,observed_values[0],rtol=0,atol=1e-10):
        revised_periods+=1
quality={"schema_version":1,"status":"source_date_vintage_qualified_for_bounded_diagnostic_information_only",
    "source_plan_sha256":sha(card/"plan.json"),"source_quality_plan_sha256":sha(card/"source_quality_plan.json"),
    "raw_zip_sha256":sha(raw),"csv_sha256":hashlib.sha256(csv_bytes).hexdigest(),
    "vintage_dates":len(release),"observation_months":len(observation),
    "latest_months_unique":release.latest_observation_month.nunique(),
    "missing_as_latest_observation_month":missing_latest,
    "positive_latest_values":int(release.latest_value.gt(0).sum()),
    "same_vintage_mom_values":int(release.mom_pct_same_vintage.notna().sum()),
    "same_vintage_yoy_values":int(release.yoy_pct_same_vintage.notna().sum()),
    "observation_months_revised_in_later_vintages":revised_periods,
    "shutdown_2025_10_first_appears_with_2025_11_at_2026_01_14":True,
    "BLS_cancellation_source":plan["gap_cause_official"],
    "calendar_sessions":len(calendar),"first_calendar_session":str(calendar[0].date()),
    "last_calendar_session":str(calendar[-1].date()),
    "all_vintages_mapped_to_next_session":len(available),
    "historical_intraday_first_receipt_qualified":False,
    "historical_market_independence":False,
    "factor_information_test_run":False,
    "market_outcome_reads":0,"model_fits":0,"accounts":0,
    "limitations":plan["limits"]}
tracking=card/"qlib_recorder"
with R.start(experiment_name=intake["experiment"],uri=str(tracking)):
    recorder=R.get_recorder()
    recorder.save_objects(artifact_path="sources",vintage_panel=frame,
        download_receipt=receipt)
    recorder.log_artifact(str(raw),artifact_path="sources")
    recorder.save_objects(artifact_path="analysis",release_availability=release,source_quality=quality)
    recorder.log_metrics(vintage_dates=len(release),observation_months=len(observation),
        same_vintage_yoy_values=quality["same_vintage_yoy_values"],market_outcome_reads=0,
        model_fits=0,accounts=0)
    quality.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id),
        recorder_uri=str(tracking),recorder_local_dir=recorder.get_local_dir())
artifacts=Path(quality["recorder_local_dir"])/"artifacts"
manifest={"recorder_id":quality["recorder_id"],"entries":[{"path":str(path.relative_to(card)),
    "sha256":sha(path),"bytes":path.stat().st_size} for path in sorted(artifacts.rglob("*")) if path.is_file()]}
(card/"verification_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
(card/"source_quality.json").write_text(json.dumps(quality,indent=2)+"\n")
summary={"experiment":intake["experiment"],"status":quality["status"],"decision":quality["status"],
    "source_quality":"source_quality.json","recorder_id":quality["recorder_id"],
    "experiment_id":quality["experiment_id"],"recorder_uri":quality["recorder_uri"],
    "recorder_local_dir":quality["recorder_local_dir"],"model_fits":0,"accounts":0,
    "qualified_forward_days":0,"objective_achieved":False,
    "artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml",
        "recorder_only_artifacts":["sources/vintage_panel","sources/vintage_panel.zip",
            "sources/download_receipt","analysis/release_availability","analysis/source_quality"]}}
(card/"summary.json").write_text(json.dumps(summary,indent=2)+"\n")
(card/"conclusion.md").write_text("# ALFRED 半导体 PPI 来源结论\n\n下载并验证 93 个日期版本、105 个月观察值；用同一版本中的数值构造 "+str(quality["same_vintage_mom_values"])+" 个环比和 "+str(quality["same_vintage_yoy_values"])+" 个同比候选，仅作来源审核。2025 年 10 月 PPI 公告因美国政府拨款中断而取消，该月数值直到 2026 年 1 月 14 日版本才与 11 月一起出现；未提前填补。所有版本只在严格晚于存档日期的下一个 Qlib 交易日 09:46 视作研究可用。\n\n日期级 ALFRED 版本与来源修订得到保留，但无法证明每个历史版本盘中首次接收。美国质量调整后的生产者价格也不是全球芯片销售或 SOXX 收益。零市场结果读取、零模型拟合、零账户；只允许另行冻结有限的信息增量检验，不能声称跑赢 SOXX。Qlib Recorder `"+quality["recorder_id"]+"`。\n")
print(json.dumps({"status":quality["status"],"vintages":len(release),"same_vintage_yoy_values":quality["same_vintage_yoy_values"],"missing_latest":missing_latest,"recorder_id":quality["recorder_id"]}))


## 中文结论

来源资格结果以同卡 `conclusion.md`、`source_quality.json`、`summary.json` 和原生 Qlib Recorder 为准。无模型或账户证据。
